<a href="https://colab.research.google.com/github/AdamKimhub/Msproject1/blob/main/forfinal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys

if "google.colab" in sys.modules:
    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
    original_data = '/content/drive/My Drive/original_dataset'
    final_data = '/content/drive/My Drive/Final_Dataset'

    # Install required packages
    !pip install pymatgen torch_geometric mp_api

else:
    original_data = "original_dataset"
    final_data = "Final_Dataset"   

In [ ]:
import pandas as pd
from pymatgen.core import Structure
import to_graph
import combine
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch_geometric.data import Data


## Data to graphs

In [ ]:
comb_df = pd.read_csv(f"{final_data}/combined/combined_data.csv")
comb_df.head()

In [ ]:
# Get first 1000 data points
comb_df = comb_df.head(1000)

In [ ]:
min_band_gap = comb_df["band_gap_value"].min()
max_band_gap = comb_df["band_gap_value"].max()
print("Before scaling:")
print(f"Min band gap: {min_band_gap}, Max band gap: {max_band_gap}")

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
comb_df["band_gap_value"] = scaler.fit_transform(comb_df[["band_gap_value"]])


min_band_gap = comb_df["band_gap_value"].min()
max_band_gap = comb_df["band_gap_value"].max()
print("After scaling:")
print(f"Min band gap: {min_band_gap}, Max band gap: {max_band_gap}")

In [ ]:
# Split the data
from sklearn.model_selection import train_test_split

train_set, test_set = train_test_split(comb_df, test_size=0.35, stratify=comb_df['strata'], random_state=42)
test_set, val_set = train_test_split(test_set, test_size=0.5, random_state=42)

In [ ]:
from sklearn.preprocessing import StandardScaler

# Create graph representation of the structures
def graphy(row):
    defective_structure = Structure.from_file(f"{original_data}/{row["dataset_material"]}/cifs/{row["_id"]}.cif")
    reference_structure = Structure.from_file(f"{final_data}/ref_cifs/{row["dataset_material"]}.cif")

    defects_only_structure = to_graph.get_defects_structure(defective_structure, reference_structure)

    nodes, edges, edge_features, ids, ratios = to_graph.get_c_graph(defects_only_structure)

    # Scale the target variable
    target = row["band_gap_value"]

    the_data = Data(
        x=torch.tensor(nodes, dtype=torch.float),
        edge_index=torch.tensor(edges, dtype=torch.long),
        edge_attr=torch.tensor(edge_features, dtype=torch.float),
        the_ids = torch.tensor(ids, dtype=torch.long).unsqueeze(0),
        the_ratios = torch.tensor(ratios, dtype=torch.float).unsqueeze(0),
        y=torch.tensor(target, dtype=torch.float).unsqueeze(0)
    )
    return the_data

# Turn each dataset into graph data
train_graphs = train_set.apply(lambda row: graphy(row), axis = 1).tolist()
val_graphs = val_set.apply(lambda row: graphy(row), axis = 1).tolist()
test_graphs = test_set.apply(lambda row: graphy(row), axis = 1).tolist()


In [ ]:
# Create data loaders
from torch_geometric.loader import DataLoader

train_loader = DataLoader(train_graphs, batch_size=1, shuffle=True)
val_loader = DataLoader(val_graphs, batch_size=1, shuffle=False)
test_loader = DataLoader(test_graphs, batch_size=1, shuffle=False)

In [ ]:
# import dependancies
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import global_mean_pool, NNConv


In [ ]:
class GNNModel(nn.Module):
    def __init__(self, node_dim=8, edge_dim=3, hidden_dim=32, embed_dim=64):
        super().__init__()
        # Edge NN maps edge_attr to weight matrix
        self.edge_nn = nn.Sequential(
            nn.Linear(edge_dim, 32),
            nn.ReLU(),
            nn.Linear(32, node_dim * hidden_dim)
        )

        self.element_embedding = nn.Embedding(118, embed_dim)

        self.conv1 = NNConv(node_dim, hidden_dim, self.edge_nn, aggr='mean')

        self.fc0 = nn.Linear(hidden_dim + embed_dim, 64)
        self.fc1 = nn.Linear(64, 32)
        self.fc2 = nn.Linear(32, 1)

    def get_u(self, the_ids, the_ratios):
        emb = self.element_embedding(the_ids)
        weighted_emb = emb * the_ratios.unsqueeze(1)
        return weighted_emb.sum(dim=0)

    def forward(self,data):
        x = F.relu(self.conv1(data.x, data.edge_index, data.edge_attr))
        x = global_mean_pool(x, data.batch)
        idss = data.the_ids.squeeze(0)
        ratioss = data.the_ratios.squeeze(0)
        global_attr = self.get_u(idss.to(self.element_embedding.weight.device),
                                 ratioss.to(self.element_embedding.weight.device)
                                 ).expand(x.size(0), -1)

        x = torch.cat([x, global_attr], dim=1)
        x = F.relu(self.fc0(x))
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

class LogCoshLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, input, target):
        ey_t = input-target
        return torch.mean(torch.log(torch.cosh(ey_t + 1e-12)))

# Instantiate model, optimizer, loss
# run the model in the gpu if the device has one
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Instance of the model
GNN_model = GNNModel().to(device)

# optimizer
optimizer = torch.optim.Adam(GNN_model.parameters(), lr=0.001)

# Loss function
# loss_fn = nn.MSELoss()
loss_fn = LogCoshLoss()

In [ ]:
def train(model):
    model.train()
    total_loss = 0
    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()

        out = model(data)
        out = scaler.inverse_transform(out.cpu().detach().numpy().reshape(-1, 1)).flatten()
        data_y = scaler.inverse_transform(data.y.cpu().detach().numpy().reshape(-1, 1)).flatten()
        loss = loss_fn(out, data_y)

        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.num_graphs
    return total_loss / len(train_loader.dataset)

def evaluate(model, loader):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for data in loader:
            data = data.to(device)

            out = model(data)
            out = scaler.inverse_transform(out.cpu().numpy().reshape(-1, 1)).flatten()
            data_y = scaler.inverse_transform(data.y.cpu().numpy().reshape(-1, 1)).flatten()
            loss = loss_fn(out, data_y)

            total_loss += loss.item() * data.num_graphs
    return total_loss / len(loader.dataset)

def predict(model, loader, actuals=False):
    model.eval()
    if actuals:
        predictions = []
        actual = []
        with torch.no_grad():
            for data in loader:
                data = data.to(device)

                out = model(data)
                out = scaler.inverse_transform(out.cpu().numpy().reshape(-1, 1)).flatten()
                data_y = scaler.inverse_transform(data.y.cpu().numpy().reshape(-1, 1)).flatten()
                predictions.append(out)
                actual.append(data_y)

        predictions = np.array(predictions).flatten()
        actual = np.array(actual).flatten()
        return predictions, actual
    else:
        predictions = []
        with torch.no_grad():
            for data in loader:
                data = data.to(device)

                out = model(data)
                out = scaler.inverse_transform(out.cpu().numpy().reshape(-1, 1)).flatten()
                predictions.append(out)

        return np.array(predictions).flatten()

In [ ]:
# Train model
the_model = GNN_model
train_losses = []
val_losses = []
for epoch in range(1, 21):
    train_loss = train(the_model)
    val_loss = evaluate(the_model,val_loader)
    print(f'Epoch {epoch:03d}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')
    train_losses.append(train_loss)
    val_losses.append(val_loss)

# Test model
test_loss = evaluate(the_model, test_loader)
print(f'Test Loss: {test_loss:.4f}')

In [ ]:
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training  and Validation Loss')
plt.legend()
plt.show()

In [ ]:
# Predict values
predicted_values, actual_values = predict(the_model, test_loader, actuals=True)

In [ ]:
# Produce a graph
plt.scatter(actual_values, predicted_values )
plt.plot(actual_values, actual_values, c="black")
plt.xlabel('Actual Band Gaps(eV)')
plt.ylabel('Predicted Band Gaps(eV)')
plt.title('Actual vs Predicted Band Gap Values')
plt.show()